# Lab 27: AIoT, the Jetson Brain and the ESP32-S3 Body

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
**AIoT** means AI plus the Internet of Things. A good AIoT product splits the work between two computers:

- The **Jetson is the brain**. It has the GPU. It sees, hears and thinks (Labs 6 to 26).
- The **ESP32-S3 is the body**. It is a small, cheap board with Wi-Fi. It switches things (LED, buzzer, servo, relay) and reads sensors, close to where the action is.

They talk with **MQTT**, a light messaging system. The Jetson does **only the AI**, and the ESP32 does **only the IoT**: this is why the Jetson's own pins are not used in this course. In this lab you learn MQTT, build the link between the two, and make an **AI alert box**: the Jetson counts the people in the camera picture and, when the room is too full, it tells the ESP32 to light the LED and sound the buzzer.

> **No ESP32 yet? No problem.** The lab gives you a **virtual ESP32** that is a Python program with the same topics and the same rules as the real board program. You build and test everything with it. When the real board is on your table, you switch it on and **nothing else changes**.

**By the end of this lab you can:**
1. Explain publish, subscribe, topics, retained messages and the last will in MQTT.
2. Send commands to a board and read its status, sensors and acknowledgements.
3. Measure the delay of a command and detect when the board goes offline.
4. Explain the fail-safe of the board and why it is needed.
5. Connect an AI result (a people count) to a device action, and send commands only when the situation changes.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. The live video appears under the cell that starts it. Press **Ctrl + S** often.

## 2. Background

**MQTT.** MQTT does not connect two programs directly. Every program connects to a **broker**, a post office. Your Jetson runs a **private broker for each student** (you saw its details in the *Details* of your student card on the teacher page). A program can:
- **publish** a message to a **topic** (the address, like `esp/cmd/led`), and
- **subscribe** to topics to receive the messages that others publish to them.

| Word | Meaning |
|---|---|
| Topic | A text address with levels separated by `/`, for example `esp/sensor/light` |
| Wildcards | `+` is any one level (`esp/+/led`), `#` is everything below (`esp/#`) |
| Payload | The message itself, here a short text such as `on` or `120` |
| Retained message | The broker keeps the last message of a topic and gives it to every new subscriber (good for a status) |
| Last will | A message that the broker publishes **for a device that disappears** (the board switches off, the Wi-Fi drops). Our board sets `esp/status = offline` as its last will. |
| QoS | How hard the broker tries to deliver: 0 at most once, 1 at least once, 2 exactly once. This lab uses QoS 0 (fast, enough on a local network). |

**Our protocol.** The Jetson and the board follow one simple agreement:

| Topic | Who publishes | Payload |
|---|---|---|
| `esp/cmd/led` | Jetson | `on`, `off` or `blink` |
| `esp/cmd/buzzer` | Jetson | milliseconds to sound |
| `esp/cmd/servo` | Jetson | angle 0 to 180 |
| `esp/cmd/alert` | Jetson | `on` or `off` (LED fast blink and buzzer beeps) |
| `esp/ack` | Board | `led=on` (the board confirms each command) |
| `esp/status` | Board | `online` or `offline` (retained, the last will) |
| `esp/telemetry` | Board | JSON every 5 seconds (uptime, signal strength, outputs) |
| `esp/sensor/light` | Board | the light level every 2 seconds |
| `esp/event` | Board | messages such as errors and the fail-safe |
| `jetson/hb` | Jetson | the **heartbeat** every 5 seconds |

**Fail-safe.** What if the Jetson crashes while the alarm is sounding? The buzzer would sound for ever. So the board has a rule: if the alert is on and **no heartbeat arrives for 15 seconds**, the board switches the alert **off** by itself. A device should always go to a **safe state** when its controller is gone.

**Send only changes.** A people counter calculates 30 times a second. It must **not** send 30 messages a second. It sends a command only when the situation **changes** (the room becomes full, or it is free again). This keeps the network quiet and the board calm. A **hold time** (like Labs 12 and 16) stops the alert from flickering, and a **hysteresis** (a different limit to switch on and off) does too.

**Latency.** The time from sending a command to the acknowledgement of the board. On a local network it is a few milliseconds.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(27)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: MQTT by Hand (35 min)

### Step 1: Your Broker
Each student has a private broker on the Jetson, with a user name (your name), a password (your PIN) and a port. The cell shows the details (not the password). Your ESP32 board will need the same details.

In [ ]:
import csv
import glob
import json
import os
import statistics
import threading
import time
from datetime import datetime

import cv2
import numpy as np

import labview

info = labview.mqtt_info()
print({k: v for k, v in info.items() if k != "password"})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Host, port and user of your broker:
- Why does each student have an own broker?:


### Step 2: Publish and Subscribe
The cell makes two clients: a **listener** that subscribes to `lab/#` and a **sender** that publishes. `labview.mqtt(on_message, subscribe)` connects to your broker and returns the client. Every message that arrives is added to the list `got`.

In [ ]:
got = []
listener = labview.mqtt(lambda topic, text: got.append((topic, text)), subscribe=["lab/#"])
sender = labview.mqtt()
time.sleep(0.5)

sender.publish("lab/hello", "Hi from the sender")
sender.publish("lab/sensor/temp", "28.5")
sender.publish("other/topic", "nobody listens to this one")
time.sleep(0.5)
for topic, text in got:
    print(f"{topic:<20} {text}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which messages did the listener get?:
- Why did it not get `other/topic`?:
- What does the `#` in `lab/#` mean?:


### Step 3: Retained Messages
A **retained** message is kept by the broker. A listener that connects **later** still receives it. This is how a device status works: you see `online` even if you connect after the device did.

In [ ]:
sender.publish("lab/status", "online", retain=True)          # retained
sender.publish("lab/normal", "not retained")                   # normal
time.sleep(0.5)

late = []
latecomer = labview.mqtt(lambda topic, text: late.append((topic, text)), subscribe=["lab/status", "lab/normal"])
time.sleep(0.8)
print("The latecomer got:", late)
sender.publish("lab/status", "", retain=True)                  # an empty retained message deletes it

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which message did the latecomer get? Why?:
- Name one thing in a real IoT system that should be retained.:


Close the three clients before you go on.

In [ ]:
for client in (listener, sender, latecomer):
    client.loop_stop()
    client.disconnect()
print("Closed.")

---
# Part B: The Link to the Board (45 min)

### Step 1: The Virtual Board
`labview.VirtualEsp32()` is a pretend ESP32 board written in Python. It subscribes to `esp/cmd/#` and `jetson/hb`, switches its pretend outputs, sends `esp/ack`, `esp/sensor/light` and `esp/telemetry`, and has the fail-safe. `labview.Esp32()` is the Jetson side of the link. Look at the code of both to see how short a protocol can be.

In [ ]:
import inspect
print(inspect.getsource(labview.Esp32.send))
print(inspect.getsource(labview.VirtualEsp32._on_message))

In [ ]:
board = labview.VirtualEsp32()          # the pretend board (replace it with the real one in Part D)
esp = labview.Esp32()                   # the Jetson side of the link
time.sleep(1)
print("Board online:", esp.online(), "| status:", esp.status)
print("Board:", board.describe())

### Step 2: Send Commands
Send commands to the board and look at what it does. `board.describe()` shows the pretend outputs. With a real board you would see the LED, hear the buzzer and see the servo move.

In [ ]:
for name, value in (("led", "on"), ("led", "blink"), ("servo", 150), ("buzzer", 500), ("led", "off"), ("servo", 30)):
    esp.send(name, value)
    time.sleep(0.3)
    print(f"sent {name:<7} {str(value):<6} ->  {board.describe()}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What did the board do for each command?:
- What happens when you send a wrong value, for example `esp.send('servo', 'abc')`? Where do you see the message (look at `esp.events`)?:


### Step 3: Measure the Latency
Each command is answered with an acknowledgement. `esp.latencies` keeps the time between the command and its answer. The cell sends 30 commands and shows the statistics.

In [ ]:
esp.latencies.clear()
for i in range(30):
    esp.send("servo", 60 + (i % 2) * 60)
    time.sleep(0.1)
time.sleep(0.5)
ms = [1000 * x for x in esp.latencies]
print(f"{len(ms)} acknowledgements | average {statistics.mean(ms):.1f} ms | slowest {max(ms):.1f} ms | fastest {min(ms):.1f} ms")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Average and slowest latency:
- Is the latency small enough for an alarm? For a robot arm?:
- Why would a real ESP32 on Wi-Fi be slower than the virtual one?:


### Step 4: Is the Board Alive?
The board publishes `esp/status` and the telemetry every 5 seconds. `esp.online()` is true only if the status is `online` **and** something was heard in the last 15 seconds. Switch the virtual board off and on and watch.

In [ ]:
print("Before:", esp.online(), esp.status)
board.stop()                                  # the board disappears: it publishes offline (the last will of a real board)
time.sleep(0.5)
print("Board stopped:", esp.online(), esp.status)
board = labview.VirtualEsp32()                # the board comes back
time.sleep(1)
print("Board back:", esp.online(), esp.status)
print("Telemetry:", esp.telemetry or "(comes every 5 seconds)")
time.sleep(4.5)
print("Telemetry:", esp.telemetry)
print("Light sensor:", esp.sensors)

### Step 5: The Fail-Safe
Switch the alert on. Then pretend the Jetson stopped: stop its heartbeat (`esp._stop.set()`) and make the board think the last heartbeat was 20 seconds ago. The board must switch the alert off by itself. (With the real board you can simply unplug the Jetson's network cable or stop the notebook.)

In [ ]:
esp.send("alert", "on")
time.sleep(0.3)
print("Alert on:  ", board.describe())
esp._stop.set()                               # stop the heartbeat of the Jetson
board.hb -= 20                                # the last heartbeat was 20 seconds ago
time.sleep(0.5)
print("After 15 s without heartbeat:", board.describe())
print("Event:", esp.events[-1][1])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What did the board do when the heartbeat stopped?:
- Why is this important for a buzzer, a door or a heater?:
- What would happen without the fail-safe?:


In [ ]:
esp._stop.clear()
threading.Thread(target=esp._heartbeat, daemon=True).start()          # start the heartbeat of the Jetson again
print("Heartbeat is running again.")

---
# Part C: The AI Alert Box (50 min)

### Step 1: The Decision
The Jetson counts people. It must decide **when to send a command**. The function `decide` uses:
- **two limits** (hysteresis): the alert turns **on** at `LIMIT` people or more, and **off** only when the count is below `LIMIT` (here `LIMIT - 1` or less, for at least `HOLD` seconds). So one person at the door does not make the alert flicker.
- a **hold time**: the new situation must last `HOLD` seconds.
- it returns `"on"` or `"off"` **only when the situation changes**, otherwise `None`.

The cell checks the logic with a made-up list of counts.

In [ ]:
# ---------------- Settings you can change ----------------
LIMIT = 3             # the alert is on when this many people or more are in view
HOLD = 2.0            # seconds a new situation must last before the command is sent
CONFIDENCE = 0.40     # confidence limit of the detector
MODEL = "yolo26n_fp16.engine"
# ----------------------------------------------------------
run = {"alert": False, "candidate": None, "since": 0.0, "sent": 0, "note": ""}


def decide(count, now):
    """Return "on" or "off" when the alert must change, otherwise None."""
    wanted = count >= LIMIT
    if wanted == run["alert"]:
        run["candidate"] = None
        return None
    if run["candidate"] != wanted:
        run["candidate"], run["since"] = wanted, now
        return None
    if now - run["since"] >= HOLD:
        run.update(alert=wanted, candidate=None)
        return "on" if wanted else "off"
    return None


run.update(alert=False, candidate=None)
counts = [0, 1, 3, 3, 4, 2, 3, 3, 3, 3, 1, 1, 1, 1, 0]            # one count per second
events = [(t, decide(c, float(t))) for t, c in enumerate(counts)]
print("Counts:  ", counts)
print("Commands:", [(t, e) for t, e in events if e])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- At which seconds was a command sent?:
- Why was no command sent at second 2 (count 3) and at second 5 (count 2)?:
- How many messages would be sent if the program sent the count of every picture (30 per second) for 1 minute? How many does `decide` send for the list above?:


### Step 2: Load the Model
The people counter uses the detector of Lab 15 (the FP16 engine). If the engine is missing, `yolo26n.pt` is used (slower).

In [ ]:
import shutil
import torch
from ultralytics import YOLO

for source in ["../lab15/yolo26n_fp16.engine", "../lab16/yolo26n_fp16.engine", "../lab15/yolo26n.pt", "../models/yolo26n.pt"]:
    name = os.path.basename(source)
    if os.path.exists(source) and not os.path.exists(name):
        shutil.copy(source, name)
if not os.path.exists(MODEL):
    print(f"{MODEL} not found, using yolo26n.pt instead (slower)")
    MODEL = "yolo26n.pt"
device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(MODEL, task="detect")
print("Using model:", MODEL)

### Step 3: The Alert Box Live
The live program counts the people (class 0), asks `decide`, and sends `esp.send("alert", ...)` only when the situation changes. It writes each change in `alertbox_log.csv`. The picture shows the count, the alert state, the link status (is the board online?), the last latency and what the board does. **If the board is offline, the program does not send** (and says so).

Run the cell, walk in front of the camera with a classmate or show a picture of people on a phone. Press **Stop** when you are done.

In [ ]:
def log_change(command, count, ok):
    first = not os.path.exists("alertbox_log.csv")
    with open("alertbox_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "command", "people", "board_online", "latency_ms"])
        latency = round(1000 * esp.latencies[-1], 1) if esp.latencies else ""
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), command, count, ok, latency])


def process(frame, state):
    result = model(frame, conf=state["Confidence"], device=device, classes=[0], verbose=False)[0]
    shown = result.plot()
    count = len(result.boxes)
    command = decide(count, time.time())
    if command:
        ok = esp.online()
        if ok:
            esp.send("alert", command)
            run["sent"] += 1
        run["note"] = f"alert {command} {'sent' if ok else 'NOT sent: board offline'} at {datetime.now().strftime('%H:%M:%S')}"
        log_change(command, count, ok)
    colour = (0, 0, 255) if run["alert"] else (0, 200, 0)
    cv2.putText(shown, f"People: {count} (limit {LIMIT})   ALERT {'ON' if run['alert'] else 'off'}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, colour, 2)
    link = "board ONLINE" if esp.online() else "board OFFLINE"
    latency = f"{1000 * esp.latencies[-1]:.0f} ms" if esp.latencies else "-"
    cv2.putText(shown, f"{link}   commands sent {run['sent']}   last latency {latency}", (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    if run.get("virtual", True) and esp.online():                    # the pretend board can say what it does; a real board is seen and heard
        cv2.putText(shown, board.describe(), (10, 82), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    if run["note"]:
        cv2.putText(shown, run["note"][:75], (10, shown.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    return shown


run.update(alert=False, candidate=None, sent=0, note="")
esp.send("alert", "off")
live = labview.live(process, controls={"Confidence": (0.05, 0.95, 0.05, CONFIDENCE)})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What did the board do when the people count went up to the limit?:
- How many commands were sent and how many pictures were processed (FPS x seconds)? Why is the difference so big?:
- What did the program do when you switched the virtual board off (run `board.stop()` in another cell, then walk in)?:


After you pressed Stop, see the log:

In [ ]:
esp.send("alert", "off")
if os.path.exists("alertbox_log.csv"):
    with open("alertbox_log.csv") as f:
        print(f.read())
else:
    print("No changes yet.")

---
# Part D: The Real ESP32-S3 (30 min, when the board is on your table)

Until now the board was pretend. This part connects the **real** board. The program that runs on it, `esp32/main.py`, follows the same protocol as the virtual board, so **the Jetson program does not change**. If you do not have the board yet, read this part and do it later.

### Step 1: Make the Board Files
The cell writes `esp32/main.py` and `esp32/config.py` in your folder. The MQTT details of **your broker** are written in `config.py` for you. Type the name and password of the lab Wi-Fi (the ESP32 uses **2.4 GHz** Wi-Fi only).

In [ ]:
WIFI_NAME = ""          # the name of the lab Wi-Fi (2.4 GHz)
WIFI_PASSWORD = ""      # its password. Do not share this notebook with the password inside.
labview.esp32_files(WIFI_NAME, WIFI_PASSWORD)
print(open("esp32/config.py").read())

### Step 2: Wire the Board
The pins in `config.py` are **placeholders**. You can use any free pins: change the numbers in `config.py` to the pins you wire. Do **not** use pins that your board needs for the USB, the flash or the boot button (see the pinout of your board).

| Part | Wiring | Pin name in `config.py` |
|---|---|---|
| LED | pin, then a 220 ohm resistor, then the LED (long leg to the resistor), then GND | `PIN_LED` |
| Active buzzer | `+` to the pin, `-` to GND | `PIN_BUZZER` |
| Servo (a door or a gate) | signal (orange) to the pin, red to 5 V, brown to GND | `PIN_SERVO` |
| Light sensor (LDR) | 3.3 V, then the LDR, then the pin, and a 10 k resistor from the pin to GND | `PIN_LIGHT` (an analog pin) |

> **Safety.** Switch the board off (unplug USB) before you change the wiring. A servo needs more current than a USB port of a laptop may give: use a good cable or a separate 5 V supply, and join the grounds.

### Step 3: Put the Program on the Board
1. Install **MicroPython** on the ESP32-S3 (once): download the firmware for your board from https://micropython.org/download/ and follow the steps of the page (it uses `esptool`), or use **Thonny** (Run, Configure interpreter, Install or update MicroPython).
2. Open **Thonny** on your laptop, choose the interpreter *MicroPython (ESP32)*, and the USB port of the board.
3. Download `esp32/main.py` and `esp32/config.py` from the file browser of JupyterLab (right-click, Download). Open them in Thonny and use **File, Save as, MicroPython device** to save both on the board with the same names.
4. Press the reset button. The board connects to the Wi-Fi and to your broker.

### Step 4: Switch from the Virtual Board to the Real One
Stop the virtual board. The real one takes over: `esp.online()` becomes true when it publishes `online`.

In [ ]:
board.stop()
run["virtual"] = False                       # the program does not read the pretend board any more
time.sleep(0.5)
print("Virtual board stopped.")
for second in range(20):
    if esp.online():
        break
    time.sleep(1)
print("Real board online:", esp.online(), "| status:", esp.status)
print("Telemetry:", esp.telemetry)

Now repeat Step 2 and Step 3 of Part B (the commands and the latency) and Step 3 of Part C (the live alert box). The cell that prints `board.describe()` will not work any more (the real board is not a Python object), so you **see and hear** the real LED, buzzer and servo instead.

In [ ]:
esp.latencies.clear()
for name, value in (("led", "on"), ("buzzer", 300), ("servo", 150), ("servo", 30), ("led", "off")):
    esp.send(name, value)
    time.sleep(1)
print("Latencies (ms):", [round(1000 * x, 1) for x in esp.latencies])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Is the board online? Which signal strength (rssi) does the telemetry show?:
- Latency with the real board compared with the virtual one:
- Did the real LED, buzzer and servo do what you sent? What did you change in the wiring or in `config.py`?:


---
# Part E: Make It Your Own, Test and Present (20 min)

### Step 1: Make It Your Own
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run the live cell again.

- Use another **AI trigger**: a keyword from Lab 19 (`start` sends `led on`), a QR code from Lab 8 (opens the servo), a missing item from Lab 16 (`alert on`).
- Use the **light sensor**: when `esp.sensors["light"]` is dark, ask the Jetson to send `led on`. (The Jetson decides, the board only obeys.)
- Show a **traffic light**: three levels, 0 to 1 people green, 2 yellow (led `blink`), 3 or more red (alert).
- Send the **count itself** to your phone on `alertbox/people` with `labview.mqtt()`, and show it in the IoT MQTT Panel app.
- Draw the **latency** of the last 30 commands as a bar chart with OpenCV (Lab 6).
- Add a command `esp/cmd/servo` that opens a gate for 5 seconds and then closes it.

> **Tip.** Change one thing at a time. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change?:
- Did it work the first time? What went wrong?:


### Step 2: Test
Test each situation and write what happened.

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | Nobody in view | Alert off, no commands | |
| 2 | People walk in until the limit | One `alert on` after about `HOLD` seconds | |
| 3 | One person leaves for 1 second, then returns | No change (hold time) | |
| 4 | Everybody leaves | One `alert off` after about `HOLD` seconds | |
| 5 | The board is offline (stop it) and the limit is reached | Not sent, logged with `False` | |
| 6 | The board comes back | `online` again | |
| 7 | Stop the notebook while the alert is on | The board switches the alert off after 15 s (fail-safe) | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 4:
- Test 5 and 6:
- Test 7: how long did the fail-safe take?:
- What else could go wrong in a real installation (Wi-Fi, power, wrong wiring)? How would your system show it?:


### Step 3: Report and Present
Fill in the report. Then give a **3-minute demonstration**: show the commands and the board reaction, the latency, the AI alert box with the board, and the fail-safe.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your design: which part is AI (Jetson) and which part is IoT (ESP32)?:
- Latency and the number of commands sent:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Where next.** In Lab 28 the Jetson opens a **door** for the right person (QR code, voice, or a face of your choice), and the ESP32 moves the servo and measures. In Lab 29 the board sends **sensor data** and the Jetson finds unusual values with AI and shows a dashboard.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The listener gets the messages of `lab/#` and not those of `other/topic`; the retained message is explained | |
| 2 | The virtual board follows the commands (LED, buzzer, servo) and wrong values are reported | |
| 3 | The latency of 30 commands is measured | |
| 4 | The board goes offline and online and `esp.online()` follows | |
| 5 | The fail-safe is shown and explained | |
| 6 | The `decide` check shows commands only when the situation changes | |
| 7 | The live alert box sends `alert on` and `alert off` and writes the log | |
| 8 | At least one improvement works, the test table is complete, and (with a real board) the real LED, buzzer and servo work | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. Which computer does the AI and which does the IoT in this lab? Why this split?:
- 2. What are a broker, a topic, publish and subscribe?:
- 3. What is a retained message and what is a last will? Give an example of each.:
- 4. Why does the program send a command only when the situation changes?:
- 5. What is the fail-safe of the board and why is it needed?:
- 6. What is hysteresis? How does it help the alert box?:
- 7. A command has a latency of 400 ms. For which applications is this too slow?:
- 8. Why can the same Jetson program control the virtual board and the real board?:
- 9. Which is a good job for the ESP32 in an AIoT system? (a) Running a language model (b) Switching a relay and reading a sensor next to the machine (c) Training a detector (d) Recording a 4K video:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `No MQTT broker` | Your lab is not started, or the kernel started before the lab | Ask your teacher to start the lab, then restart this notebook's kernel |
| The listener gets nothing | The topic does not match, or you subscribed after the message | Check the topic spelling. Use `#` to see everything. Wait half a second after `subscribe`. |
| `esp.online()` is always `False` | The board (virtual or real) is not running, or it is not connected to **your** broker | Run the virtual board cell. For the real board check `config.py` and the telemetry in a terminal. |
| The commands work but there is no acknowledgement | The board is not the one that follows our protocol | Check the topics in the table of the Background section |
| Messages from an old session appear | Retained messages stay in the broker | Publish an empty retained message to that topic to delete it |
| The real board does not connect to the Wi-Fi | A 5 GHz network, a wrong password, or too far | Use a 2.4 GHz network. Check `WIFI_SSID` and `WIFI_PASSWORD`. Look at the output in Thonny. |
| The real board connects to Wi-Fi but not to MQTT | Wrong `MQTT_HOST`, `MQTT_PORT`, user or password | Compare with the Details of your student card. The Jetson IP can change: look at the teacher page. |
| `ImportError: umqtt` on the board | The MicroPython firmware has no `umqtt.simple` | In Thonny use Tools, Manage packages and install `micropython-umqtt.simple`, or use the standard firmware for your board |
| The servo shakes or the board resets | The servo draws too much current | Use a separate 5 V supply and join the grounds |
| The LED or the buzzer does not work | Wrong pin, wrong polarity, a pin the board uses | Check `config.py`, the polarity of the LED, and the pinout of your board |
| The alert switches off by itself after 15 s | The heartbeat of the Jetson stopped (the fail-safe) | Keep the notebook running. If you stopped the heartbeat in Part B, run the cell that starts it again. |
| `CUDA out of memory` | Other notebooks use the memory | Press Stop in other live views, Kernel, Shut Down All Kernels, run again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(27)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- MQTT: https://mqtt.org/
- MicroPython for the ESP32: https://docs.micropython.org/en/latest/esp32/quickref.html
- umqtt.simple: https://github.com/micropython/micropython-lib/tree/master/micropython/umqtt.simple
- Thonny: https://thonny.org/
- ESP32-S3 pinout: see the page of your board